# LexAgent v2.0 | Phase 1 | End-to-End Test

**DVA+ Protocol — Hybrid Retrieval Layer**

This notebook validates the complete Phase 1 pipeline:
1. Install dependencies
2. Mount Google Drive
3. Load Mistral-7B (NF4 4-bit)
4. Load Legal-BERT embedder
5. Download legal datasets (LegalBench, CaseHOLD, SCOTUS)
6. Build ChromaDB index
7. Build BM25 index
8. Test hybrid retrieval (RRF fusion)
9. Print system summary

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 1: Install Requirements
# ══════════════════════════════════════════════════════
!pip install -q -r /content/LexAgent/lexagent/requirements.txt
print("✅ All dependencies installed.")

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 2: Mount Google Drive & Setup Paths
# ══════════════════════════════════════════════════════
import sys
import os

# Add LexAgent to Python path
sys.path.insert(0, '/content/LexAgent/lexagent')

from src.utils.drive_manager import DriveManager
from config import DRIVE_BASE, CHROMA_CORPUS_PATH, BM25_INDEX_PATH

# Mount Drive and create directories
drive_mgr = DriveManager()
drive_mgr.mount()

print(f"\n✅ Drive mounted. Base path: {DRIVE_BASE}")
print(f"   ChromaDB path: {CHROMA_CORPUS_PATH}")
print(f"   BM25 index:    {BM25_INDEX_PATH}")

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 3: Load Mistral-7B (NF4 4-bit Quantization)
# ══════════════════════════════════════════════════════
import torch
from src.models.load_model import load_mistral_7b

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB")
print(f"VRAM used (before model): {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print()

model, tokenizer, llm_pipeline = load_mistral_7b()

print(f"\n✅ Mistral-7B loaded in NF4 4-bit.")
print(f"   VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 4: Load Legal-BERT Embedder (CPU)
# ══════════════════════════════════════════════════════
from src.models.load_model import load_embedder

embedder = load_embedder()

# Quick sanity check
test_embed = embedder.encode(["test query"])
print(f"\n✅ Legal-BERT embedder loaded on CPU.")
print(f"   Embedding dimension: {test_embed.shape[1]}")
print(f"   VRAM (should be same as before): {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 5: Download Datasets (Skip if Already on Drive)
# ══════════════════════════════════════════════════════
from src.data.download_datasets import download_all_datasets

print("Downloading datasets (will skip existing files)...\n")
dataset_paths = download_all_datasets(save_dir=DRIVE_BASE)

print(f"\n✅ All datasets ready:")
for name, path in dataset_paths.items():
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    print(f"   {name}: {path} ({size_mb:.1f} MB)")

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 6: Build Corpus + ChromaDB Index (Skip if Exists)
# ══════════════════════════════════════════════════════
import chromadb
from src.data.build_corpus import build_legal_corpus, build_chroma_index
from config import CORPUS_JSON_PATH, KNOWN_CASES_PATH
import json
from src.data.corpus_schema import CorpusDocument

# Check if ChromaDB index already exists
chroma_exists = os.path.exists(CHROMA_CORPUS_PATH) and os.listdir(CHROMA_CORPUS_PATH)

if chroma_exists:
    print("ChromaDB index found on Drive. Loading existing index...")
    client = chromadb.PersistentClient(path=CHROMA_CORPUS_PATH)
    collection = client.get_collection("legal_corpus")
    
    # Load corpus from JSON
    with open(CORPUS_JSON_PATH, 'r') as f:
        corpus_dicts = json.load(f)
    corpus = [CorpusDocument.from_dict(d) for d in corpus_dicts]
    
    print(f"   Loaded {collection.count()} documents from existing index.")
else:
    print("Building corpus and ChromaDB index (first run)...\n")
    
    # Build corpus from downloaded datasets
    corpus = build_legal_corpus(dataset_paths)
    
    # Build ChromaDB index with Legal-BERT embeddings
    collection = build_chroma_index(corpus, embedder)

print(f"\n✅ Corpus ready: {len(corpus)} documents")
print(f"   ChromaDB collection: {collection.count()} vectors")
print(f"   Known cases saved: {KNOWN_CASES_PATH}")

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 7: Build BM25 Index (Skip if Exists)
# ══════════════════════════════════════════════════════
from src.retrieval.bm25_retriever import BM25Retriever
from config import BM25_INDEX_PATH

if os.path.exists(BM25_INDEX_PATH):
    print("BM25 index found on Drive. Loading...")
    bm25_retriever = BM25Retriever.load_index(BM25_INDEX_PATH, corpus)
else:
    print("Building BM25 index (first run)...")
    bm25_retriever = BM25Retriever(corpus)
    bm25_retriever.save_index(BM25_INDEX_PATH)

print(f"\n✅ BM25 index ready: {len(corpus)} documents indexed")
print(f"   Saved at: {BM25_INDEX_PATH}")

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 8: Test Hybrid Retrieval with Sample Query
# ══════════════════════════════════════════════════════
from src.retrieval.dense_retriever import DenseRetriever
from src.retrieval.hybrid_retriever import HybridRetriever

# Initialize retrievers
dense_retriever = DenseRetriever(collection, embedder)
hybrid_retriever = HybridRetriever(dense_retriever, bm25_retriever)

# ── Test query ──────────────────────────────────────────
query = "Can a landlord evict a tenant without notice?"
results = hybrid_retriever.retrieve(query, jurisdiction="US_Federal")

print(f"\n{'═' * 70}")
print(f"HYBRID RETRIEVAL RESULTS")
print(f"Query: {query}")
print(f"Jurisdiction: US_Federal")
print(f"{'═' * 70}\n")

# Print top-3 results
for i, doc in enumerate(results[:3], 1):
    rrf_score = doc.metadata.get('rrf_score', 'N/A')
    print(f"── Result {i} ──")
    print(f"   Case Name:  {doc.case_name if doc.case_name else '(not extracted)'}")
    print(f"   Source:     {doc.source}")
    print(f"   Court:     {doc.court if doc.court else '(unknown)'}")
    print(f"   RRF Score: {rrf_score}")
    print(f"   Text:      {doc.text[:200]}...")
    print()

# ── Test entity map (NOVEL: feeds CCE) ───────────────────
entity_map = hybrid_retriever.get_entity_map(results)
print(f"\n── Entity Map (for CCE) ──")
print(f"   Known case names in results: {len(entity_map)}")
for case_name in list(entity_map.keys())[:5]:
    print(f"   • {case_name}")

print(f"\n✅ Hybrid retrieval working correctly!")

In [ ]:
# ══════════════════════════════════════════════════════
# Cell 9: System Summary
# ══════════════════════════════════════════════════════
import pickle

print("\n" + "═" * 70)
print(" LexAgent v2.0 — Phase 1 Summary")
print("═" * 70)

# Corpus stats
scotus_count = sum(1 for d in corpus if d.source == 'SCOTUS')
casehold_count = sum(1 for d in corpus if d.source == 'CaseHOLD')
print(f"\n📚 Corpus:")
print(f"   Total documents:    {len(corpus)}")
print(f"   SCOTUS opinions:    {scotus_count}")
print(f"   CaseHOLD holdings:  {casehold_count}")

# Index stats
print(f"\n🔍 Indices:")
print(f"   ChromaDB vectors:   {collection.count()}")
bm25_size = os.path.getsize(BM25_INDEX_PATH) / 1e6 if os.path.exists(BM25_INDEX_PATH) else 0
print(f"   BM25 index size:    {bm25_size:.1f} MB")

# Known cases
if os.path.exists(KNOWN_CASES_PATH):
    with open(KNOWN_CASES_PATH, 'rb') as f:
        known_cases = pickle.load(f)
    print(f"   Known case names:   {len(known_cases)}")

# VRAM stats
print(f"\n💾 VRAM:")
print(f"   Model VRAM:         {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   Peak VRAM:          {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
print(f"   Total GPU memory:   {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB")

# Model info
from config import MODEL_NAME, EMBEDDER_NAME
print(f"\n🤖 Models:")
print(f"   LLM:       {MODEL_NAME} (NF4 4-bit)")
print(f"   Embedder:  {EMBEDDER_NAME} (CPU)")

print(f"\n{'═' * 70}")
print(f"✅ Phase 1 complete! Ready for Phase 2 (DVA+ Debate Pipeline).")
print(f"{'═' * 70}")